*****Enocder Pretraining -- BERT with Masked Language Modelling*****

Libraries Required

In [1]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('gutenberg')

import string
import torch
import numpy as np
import random
from nltk.corpus import stopwords, gutenberg
from transformers import BertTokenizer, BertForMaskedLM
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


In [2]:
# Save Austen text to file
# The `gutenberg.raw()` function returns the raw full text of the book as a single string.
emma_text = gutenberg.raw('austen-emma.txt')

# We save the full text to a local file named "sample.txt"
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(emma_text)


# Load lines and clean
# Open the file and read only the first 1000 lines (to keep the dataset small for demo/training)
with open('sample.txt', 'r') as f:
    lines = f.readlines()[:1000]


# Load a list of English stop words from NLTK
stop_words = stopwords.words('english')

# Remove newline characters and convert all text to lowercase
lines = [line.strip().lower() for line in lines]

# Remove punctuation from each line using `str.maketrans`
# It creates a translation table that maps each punctuation character to `None`
lines = [line.translate(str.maketrans('', '', string.punctuation)) for line in lines]


# Function to remove stop words from each line
def remove_stops(text):
    # For each line in the input list:
    #   - Split the line into words
    #   - Keep only words that are not in stop_words
    #   - Join the filtered words back into a sentence
    #   - Skip any empty lines
    return [' '.join([w for w in line.split() if w not in stop_words]) for line in text if line]


# Apply the stop word removal function to the cleaned lines
cleaned_lines = remove_stops(lines)


In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') #Wordpiece .... unbelievable ---> ##un ##be ##lieve
max_len = 100
MASK_TOKEN_ID = tokenizer.mask_token_id
CLS_TOKEN_ID = tokenizer.cls_token_id
SEP_TOKEN_ID = tokenizer.sep_token_id
PAD_TOKEN_ID = tokenizer.pad_token_id


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [4]:
class MLM_Dataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=100, mask_prob=0.15):
        """
        Args:
            texts (List[str]): List of input sentences.
            tokenizer (BertTokenizer): Pretrained BERT tokenizer.
            max_len (int): Maximum length of tokenized input sequences.
            mask_prob (float): Probability of masking a token (typically 0.15 for MLM).
        """
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_len = max_len
        self.mask_prob = mask_prob

    def __len__(self):
        """Returns total number of samples."""
        return len(self.texts)

    def mask_tokens(self, input_ids):
        """
        Randomly masks tokens in the input_ids for MLM (Masked Language Modeling) task.
        Args:
            input_ids (Tensor): Tensor of token ids for one sentence.
        Returns:
            Tuple of (masked_input_ids, labels) where:
                - masked_input_ids: tokens with some replaced by [MASK] token.
                - labels: original tokens where unmasked tokens are replaced with -100 (ignored by loss).
        """
        # Clone input_ids to create labels (before masking)
        labels = input_ids.clone()

        # Create a matrix with the same shape as labels filled with `mask_prob` (e.g., 0.15)
        probability_matrix = torch.full(labels.shape, self.mask_prob)

        # Get a binary mask for special tokens ([CLS], [SEP], [PAD], etc.)
        # This ensures special tokens are NOT masked
        special_tokens_mask = self.tokenizer.get_special_tokens_mask(
            labels.tolist(), already_has_special_tokens=True  # Works on list of token IDs
        )
        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)

        # Zero out the masking probability for special tokens
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)

        # Sample tokens to mask based on the probability matrix (Bernoulli sampling)
        masked_indices = torch.bernoulli(probability_matrix).bool()

        # Set label to -100 where token is not masked (ignored in loss computation)
        labels[~masked_indices] = -100

        # Replace masked token positions with the special [MASK] token ID (103 for BERT)
        input_ids[masked_indices] = self.tokenizer.mask_token_id

        return input_ids, labels

    def __getitem__(self, idx):
        """
        Returns a single sample from the dataset, with token masking applied.
        Args:
            idx (int): Index of the sample.
        Returns:
            Dictionary with:
                - 'input_ids': masked input token ids
                - 'attention_mask': attention mask (1 for real token, 0 for padding)
                - 'labels': labels with original tokens at masked positions and -100 elsewhere
        """
        # Tokenize the sentence into input_ids and attention_mask
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,               # Cut off longer sequences
            padding='max_length',          # Pad shorter sequences to max_len
            max_length=self.max_len,
            return_tensors='pt'            # Return as PyTorch tensors
        )

        # Remove the extra batch dimension (from return_tensors='pt') → shape [max_len]
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        # Mask tokens and prepare labels
        masked_input_ids, labels = self.mask_tokens(input_ids.clone())

        return {
            'input_ids': masked_input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


In [5]:
# Create the dataset for Masked Language Modeling using cleaned lines of text
dataset = MLM_Dataset(cleaned_lines, tokenizer, max_len=max_len)

# Create a DataLoader to fetch batches of data during training
# batch_size=16 means each batch contains 16 samples
# shuffle=True ensures the data is shuffled each epoch for better training
loader = DataLoader(dataset, batch_size=16, shuffle=True)


# ---------------------- Model Setup ----------------------

# Load the pre-trained BERT model with the Masked Language Modeling head
# This model outputs logits for each token position to predict the original token
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# Set the model to training mode (enables dropout, layer norm updates, etc.)
model.train()

# Choose device: GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the selected device
model.to(device)

# Optimizer: AdamW (Adam with weight decay), commonly used for transformer training
# lr=1e-4 is a standard starting learning rate for fine-tuning BERT models
optimizer = AdamW(model.parameters(), lr=1e-4)

# Loss function: CrossEntropyLoss
# This loss is suitable for classification tasks where the model predicts token IDs
# It combines LogSoftmax and NLLLoss in one function
# Note: In MLM, labels have -100 for tokens we want to ignore in the loss computation (unmasked tokens)
loss_fn = torch.nn.CrossEntropyLoss()


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
epochs = 3  # Number of times the entire dataset will be passed through the model

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")  # Display current epoch number

    # tqdm creates a progress bar for the DataLoader loop
    loop = tqdm(loader, leave=True)

    total_loss = 0  # To accumulate loss over the entire epoch

    for batch in loop:
        # Move inputs and labels to the correct device (GPU or CPU)
        input_ids = batch['input_ids'].to(device)           # Shape: [batch_size, seq_length]
        attention_mask = batch['attention_mask'].to(device) # Shape: [batch_size, seq_length]
        labels = batch['labels'].to(device)                 # Shape: [batch_size, seq_length]

        # Forward pass: the model returns a dict including the loss when labels are provided
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        loss = outputs.loss  # Extract the computed loss from outputs

        total_loss += loss.item()  # Add current batch loss to total_loss (convert tensor to float)

        # Backpropagation: compute gradients of the loss w.r.t model parameters
        loss.backward()

        # Update model parameters based on computed gradients
        optimizer.step()

        # Reset gradients for next iteration to avoid accumulation
        optimizer.zero_grad()

        # Update tqdm progress bar description and postfix (shows loss value)
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    # Optionally, print average loss per epoch after the loop
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

    # ---------------- Save model checkpoint after each epoch ----------------
    save_path = f"bert_mlm_epoch_{epoch+1}.pt"  # Define file name
    torch.save(model.state_dict(), save_path)   # Save model weights
    print(f"Model saved to {save_path}")        # Confirm saving

print("Training complete.")



Epoch 1/3


Epoch 1: 100%|██████████| 55/55 [00:17<00:00,  3.08it/s, loss=13.3]


Epoch 1 average loss: 7.1791
Model saved to bert_mlm_epoch_1.pt

Epoch 2/3


Epoch 2: 100%|██████████| 55/55 [00:17<00:00,  3.17it/s, loss=6.38]


Epoch 2 average loss: 6.2279
Model saved to bert_mlm_epoch_2.pt

Epoch 3/3


Epoch 3: 100%|██████████| 55/55 [00:18<00:00,  2.99it/s, loss=8.87]


Epoch 3 average loss: 5.9940
Model saved to bert_mlm_epoch_3.pt
Training complete.


In [7]:
# Define saved model path (change if needed)
saved_model_path = "/content/bert_mlm_epoch_1.pt"

# Re-initialize model architecture
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# Load trained weights
model.load_state_dict(torch.load(saved_model_path, map_location=device))

# Move model to correct device
model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [8]:
def predict_masked_words(model, tokenizer, text):
    model.eval()

    # Tokenize and get inputs; move each tensor to device
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Find positions of [MASK] tokens in input_ids
    mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits  # Shape: (batch_size=1, seq_len, vocab_size)

    # Get predicted token ids at mask positions by taking argmax over vocab dimension
    predicted_token_ids = logits[0, mask_token_index].argmax(dim=-1)

    # Decode each predicted token id into words
    predicted_tokens = [tokenizer.decode([token_id]) for token_id in predicted_token_ids]

    # Join predicted tokens into a string (you can customize this join style if needed)
    return ' '.join(predicted_tokens).strip()

# Example usage:
queries = [
    "too well [MASK] for her",
    "nice to [MASK] her",
    "Emma [MASK] a girl who wanted a [MASK]"
]

for q in queries:
    prediction = predict_masked_words(model, tokenizer, q)
    print(f"{q} -> {prediction}")


too well [MASK] for her -> done
nice to [MASK] her -> see
Emma [MASK] a girl who wanted a [MASK] -> wanted thing


In [9]:
# Define saved model path (change if needed)
saved_model_path = "/content/bert_mlm_epoch_3.pt"

# Re-initialize model architecture
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# Load trained weights
model.load_state_dict(torch.load(saved_model_path, map_location=device))

# Move model to correct device
model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [10]:
def predict_masked_words(model, tokenizer, text):
    model.eval()

    # Tokenize and get inputs; move each tensor to device
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Find positions of [MASK] tokens in input_ids
    mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits  # Shape: (batch_size=1, seq_len, vocab_size)

    # Get predicted token ids at mask positions by taking argmax over vocab dimension
    predicted_token_ids = logits[0, mask_token_index].argmax(dim=-1)

    # Decode each predicted token id into words
    predicted_tokens = [tokenizer.decode([token_id]) for token_id in predicted_token_ids]

    # Join predicted tokens into a string (you can customize this join style if needed)
    return ' '.join(predicted_tokens).strip()

# Example usage:
queries = [
    "too well [MASK] for her",
    "nice to [MASK] her",
    "Emma [MASK] a girl who wanted a [MASK]"
]

for q in queries:
    prediction = predict_masked_words(model, tokenizer, q)
    print(f"{q} -> {prediction}")


too well [MASK] for her -> made
nice to [MASK] her -> see
Emma [MASK] a girl who wanted a [MASK] -> liked man
